# A5 First-System PRINT Patch Export

Reads `a5_channels.npz` from Google Drive and exports the exact frozen 33×33×11 CNN patches for final PRINT coordinates in the first-system diagnostic region.

Inputs expected in:

`/content/drive/MyDrive/MusicAnnotationProject/a5_channels/`

Required files:
- `a5_channels.npz`
- `a5_final_print_diagnostics.csv`

Output:
- `a5_first_system_print_patches.csv`
- `a5_first_system_export_manifest.json`

The tensor contract matches the V8.2 model: channels are ordered `obs, norm, mask, depth, horizontal, vertical, compact, preserve, remove, ambiguous, edge`; `obs` and `norm` are divided by 255; all channels use reflect padding; patch size is 33×33.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject/a5_channels')
CHANNEL_FILE = ROOT / 'a5_channels.npz'
PRINT_FILE = ROOT / 'a5_final_print_diagnostics.csv'
OUTPUT_FILE = ROOT / 'a5_first_system_print_patches.csv'
MANIFEST_FILE = ROOT / 'a5_first_system_export_manifest.json'

CHANNELS = [
    'obs',
    'norm',
    'mask',
    'depth',
    'horizontal',
    'vertical',
    'compact',
    'preserve',
    'remove',
    'ambiguous',
    'edge',
]

PATCH = 33
HALF = PATCH // 2

# Diagnostic first-system region from the current Ardross page.
Y_MIN = 650
Y_MAX = 1000
X_MIN = 150
X_MAX = 1250

for path in [CHANNEL_FILE, PRINT_FILE]:
    if not path.is_file():
        raise RuntimeError(f'Missing required input: {path}')

print('Channel file:', CHANNEL_FILE)
print('PRINT diagnostics:', PRINT_FILE)
print('Output:', OUTPUT_FILE)


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

print_rows = pd.read_csv(PRINT_FILE)
required_columns = {'y', 'x', 'raw_p_print', 'calibrated_p_print'}
missing = sorted(required_columns - set(print_rows.columns))
if missing:
    raise RuntimeError(
        'Final PRINT diagnostic CSV is missing required columns: '
        f'{missing}'
    )

print_rows['y'] = pd.to_numeric(print_rows['y'], errors='raise').astype(int)
print_rows['x'] = pd.to_numeric(print_rows['x'], errors='raise').astype(int)
print_rows['raw_p_print'] = pd.to_numeric(print_rows['raw_p_print'], errors='raise')
print_rows['calibrated_p_print'] = pd.to_numeric(print_rows['calibrated_p_print'], errors='raise')

original_rows = len(print_rows)
print_rows = print_rows[
    (print_rows['y'] >= Y_MIN)
    & (print_rows['y'] <= Y_MAX)
    & (print_rows['x'] >= X_MIN)
    & (print_rows['x'] <= X_MAX)
].copy()
print_rows = print_rows.sort_values(['y', 'x']).reset_index(drop=True)

if print_rows.empty:
    raise RuntimeError(
        'No final PRINT coordinates lie inside the configured first-system region. '
        'Check Y_MIN/Y_MAX/X_MIN/X_MAX against the page coordinates.'
    )

print('All final PRINT rows:', original_rows)
print('First-system final PRINT rows:', len(print_rows))
print('Region:', {'y': [Y_MIN, Y_MAX], 'x': [X_MIN, X_MAX]})


In [ ]:
with np.load(CHANNEL_FILE, allow_pickle=False) as state:
    missing_channels = [name for name in CHANNELS if name not in state.files]
    if missing_channels:
        raise RuntimeError(
            'a5_channels.npz is missing required channels: '
            f'{missing_channels}; available={state.files}'
        )

    unexpected = [name for name in state.files if name not in CHANNELS]
    if unexpected:
        print('Additional NPZ arrays ignored:', unexpected)

    arrays = {
        name: np.ascontiguousarray(np.asarray(state[name]))
        for name in CHANNELS
    }

shape = arrays['obs'].shape
if len(shape) != 2:
    raise RuntimeError(f'A5 observed channel must be 2-D; found {shape}')

for name in CHANNELS:
    if arrays[name].shape != shape:
        raise RuntimeError(
            f'A5 channel {name!r} has shape {arrays[name].shape}; '
            f'expected {shape}'
        )

print('Page shape:', shape)
print('Channel order:', CHANNELS)
print('NPZ SHA-256:', sha256_file(CHANNEL_FILE))


In [ ]:
# Reproduce the exact frozen V8.2 CNN preprocessing.
padded = {}

for name in CHANNELS:
    value = arrays[name].astype(np.float32)

    if name in {'obs', 'norm'}:
        value = value / 255.0

    padded[name] = np.pad(
        value,
        HALF,
        mode='reflect',
    )

expected_padded_shape = (
    shape[0] + 2 * HALF,
    shape[1] + 2 * HALF,
)

for name in CHANNELS:
    if padded[name].shape != expected_padded_shape:
        raise RuntimeError(
            f'Padded channel {name!r} has shape {padded[name].shape}; '
            f'expected {expected_padded_shape}'
        )
    if padded[name].dtype != np.float32:
        raise RuntimeError(
            f'Padded channel {name!r} has dtype {padded[name].dtype}; '
            'expected float32'
        )
    if not np.isfinite(padded[name]).all():
        raise RuntimeError(f'Padded channel {name!r} contains non-finite values')

print('Frozen tensor preprocessing reproduced.')
print('Patch:', PATCH, 'x', PATCH)
print('Padded shape:', expected_padded_shape)


In [ ]:
metadata_columns = [
    'y',
    'x',
    'raw_p_print',
    'calibrated_p_print',
    'provisional_decision',
    'final_decision',
    'yolo_confidence',
]

patch_columns = []
for channel in CHANNELS:
    for dy in range(-HALF, HALF + 1):
        for dx in range(-HALF, HALF + 1):
            patch_columns.append(
                f'{channel}_dy{dy:+03d}_dx{dx:+03d}'
            )

fieldnames = metadata_columns + patch_columns
print('Patch-value columns:', len(patch_columns))
print('Total CSV columns:', len(fieldnames))


In [ ]:
temporary = OUTPUT_FILE.with_name(OUTPUT_FILE.name + '.tmp')

with temporary.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()

    for _, source_row in print_rows.iterrows():
        y = int(source_row['y'])
        x = int(source_row['x'])

        if y < 0 or y >= shape[0] or x < 0 or x >= shape[1]:
            raise RuntimeError(
                f'Final PRINT coordinate ({y}, {x}) is outside page shape {shape}'
            )

        row = {
            'y': y,
            'x': x,
            'raw_p_print': float(source_row['raw_p_print']),
            'calibrated_p_print': float(source_row['calibrated_p_print']),
            'provisional_decision': source_row.get('provisional_decision', ''),
            'final_decision': source_row.get('final_decision', ''),
            'yolo_confidence': source_row.get('yolo_confidence', ''),
        }

        for channel in CHANNELS:
            patch = padded[channel][
                y:y + PATCH,
                x:x + PATCH,
            ]

            if patch.shape != (PATCH, PATCH):
                raise RuntimeError(
                    f'{channel} patch at ({y}, {x}) has shape {patch.shape}'
                )

            for py, dy in enumerate(range(-HALF, HALF + 1)):
                for px, dx in enumerate(range(-HALF, HALF + 1)):
                    column = f'{channel}_dy{dy:+03d}_dx{dx:+03d}'
                    row[column] = float(patch[py, px])

        writer.writerow(row)

temporary.replace(OUTPUT_FILE)

print('Created:', OUTPUT_FILE)
print('Rows:', len(print_rows))
print('Bytes:', OUTPUT_FILE.stat().st_size)
print('SHA-256:', sha256_file(OUTPUT_FILE))


In [ ]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'purpose': 'A5 first-system final-PRINT full 33x33x11 patch export',
    'source_npz': str(CHANNEL_FILE),
    'source_npz_sha256': sha256_file(CHANNEL_FILE),
    'source_print_diagnostics': str(PRINT_FILE),
    'source_print_diagnostics_sha256': sha256_file(PRINT_FILE),
    'output_csv': str(OUTPUT_FILE),
    'output_csv_sha256': sha256_file(OUTPUT_FILE),
    'page_shape': list(shape),
    'channel_order': CHANNELS,
    'patch_size': PATCH,
    'patch_half_width': HALF,
    'padding': 'reflect',
    'normalisation': {
        'obs': 'float32 / 255',
        'norm': 'float32 / 255',
        'other_channels': 'native float32',
    },
    'region': {
        'y_min': Y_MIN,
        'y_max': Y_MAX,
        'x_min': X_MIN,
        'x_max': X_MAX,
    },
    'all_final_print_rows': int(original_rows),
    'exported_final_print_rows': int(len(print_rows)),
    'patch_value_columns': int(len(patch_columns)),
    'total_csv_columns': int(len(fieldnames)),
}

MANIFEST_FILE.write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding='utf-8',
)

print(json.dumps(manifest, indent=2))


## Files to upload back to Copilot

Upload these two files from the Drive folder after the notebook finishes:

1. `a5_first_system_print_patches.csv`
2. `a5_first_system_export_manifest.json`

Do not delete `a5_channels.npz`; the CSV is a diagnostic extract, while the NPZ remains the authoritative full-page channel record.
